Read in the ESP monthly volumes for each month for each year for all forecast locations
Then for each month, caluclate the 10, 25 ,50, 75, and 90 percentile volumes
Then export to a .csv file (for now)
Created by P. Becker April, 2025

In [ ]:
#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as scipy
from scipy import stats
import seaborn as sns
import os
import glob
from datetime import datetime

Section below generates the ESP 10,25,50,75, and 90 traces

In [ ]:
#get the esp 5 year volumes and and convert to percentiles for each site for each forecast date (11/81 through 9/21)
#directory
# path='/perc3/data/ensts_refcst/MONLY/'

# #where to save the files, update the site folder as needed
# path2='/home/paige.becker/24MS/YDLC2/'
# #files are formated as follows: STNID.YYYY-MM-DD.espmvol.5yr.csv
# #starts with 1980-11-01 and ends 2021-09-01

# #use glob to get matching file paths
# file_pattern=os.path.join(path,'YDLC2.*.espmvol.5yr.csv')


# file_list=glob.glob(file_pattern)

# dataframes={}

# #loop through each file and read it into a dataframe
# for file in file_list:
#     #extract the date from the file name for naming dataframe
#     filename=os.path.basename(file)
#     date_str=filename.split('.')[1] #date is in second position

#     try:

#         df=pd.read_csv(file, delimiter=',',skiprows=2)
#         #convert 'date' column to datetime format
#         df['DATE']=pd.to_datetime(df['DATE'],format='%m/%Y')

#         #calculate 10th, 50th, and 90th percentile for each row
#         percentiles=df.iloc[:,1:].quantile([0.1,0.5,0.9],axis=1).T

#         percentiles.columns=['10th Percentile','50th Percentile','90th Percentile']

#         #add Date column to percentiles dataframe
#         percentiles.insert(0,'DATE',df['DATE'])

#         #save resulting dataframe to new csv file
#         output_file=os.path.join(path2,f"{date_str}_percentile.csv")
#         percentiles.to_csv(output_file,index=False)

#         #store the dataframe in the dictionary with the date as key
#         dataframes[date_str]=df

#     except Exception as e:
#         print(f"Error processing {file}: {e}")
# #display the keys in the dataframes dictionary
# print(dataframes.keys())

#the above was done 4/22/25

In [ ]:
#upload historical climatology flows from UCB
#labeled as 'HistoricalXX.csv' with the number corresponding to the percentile, so Historical10 is the 10th percentile (lowest 10%)
path='/home/paige.becker/24MS/'
hist10=pd.read_csv(path+'Historical10.csv')
hist25=pd.read_csv(path+'Historical25.csv')
hist50=pd.read_csv(path+'HistoricalMedian.csv')
hist75=pd.read_csv(path+'Historical75.csv')
hist90=pd.read_csv(path+'Historical90.csv')

print(hist50)

START HERE

In [93]:
import pandas as pd
import numpy as np
import os

## Make sure Lines 44 and 105 are updated for each site
# Function to calculate the interpolation factor
def calculate_interpolation_factor(esp_september, hist_september, hist_december):
    """
    Calculate interpolation factor based on the September ESP and Historical values,
    and December Historical values.
    """
    factor = (esp_september / hist_september) * (hist_december / hist_december)
    return factor

# Function to generate the forecast based on the rules
def generate_forecast(forecast_month, year, historical_df, esp_directory):
    """
    Generate forecast based on the forecast month, ESP data, and Powell historical data.
    
    forecast_month: Month when forecast is issued (1-12).
    year: Year of the forecast (e.g., 1981, 1982, ..., 2020).
    historical_df: Historical DataFrame containing Powell historical data.
    esp_directory: Path to the directory containing ESP files.
    """
    
    # Construct the filename for the ESP file for the given year and month
    esp_filename = f'{year}-{str(forecast_month).zfill(2)}-01_percentile.csv'
    esp_filepath = os.path.join(esp_directory, esp_filename)
    
    # Load the ESP data from the file
    try:
        esp_df = pd.read_csv(esp_filepath)
    except FileNotFoundError:
        print(f"Error: ESP file '{esp_filename}' not found.")
        return []

    # Extract the 50th Percentile ESP values
    esp = esp_df['50th Percentile'].values
    
    # Extract the historical data for chosen site
    #site options are: 'Powell', 'Fontenelle', 'Flaming Gorge', 'BlueMesa',
       #'MorrowPoint', 'Crystal ', 'TaylorPark ', 'Vallecito', 'Navajo (Mod)',
       #'Animas at Durango', 'GJ Local', 'Yampa Maybell Plus Lily'
    historical = historical_df['Yampa Maybell Plus Lily'].values
    extended_historical=np.tile(historical,2)
    forecast = []

    #month index mapping because historical starts in October
    idx={'oct': 0, 'nov': 1, 'dec': 2,
        'jan': 3, 'feb': 4, 'mar': 5,
        'apr': 6, 'may': 7, 'jun': 8,
        'jul': 9, 'aug': 10, 'sep': 11}
    #track current month index in 0-based (jan =1, dec =12)
    current_month_idx=forecast_month-1
    # For January through May (forecast_month 1 to 5)
    if forecast_month <= 5:
        # First part: ESP for corresponding month through month 9
        
        months_from_issue_to_sep=10-current_month_idx
        forecast.extend(esp[current_month_idx:months_from_issue_to_sep]) #up to ESP september but not including index 9 (spetember is index 8)
        
        # Interpolation for October and November
        esp_september = esp[8]
          # ESP for September
        #hist_september = historical[idx['sep']]  # Historical for September
        #hist_october=historical[idx['oct']]
        #hist_november=historical[idx['nov']]
        #hist_december = historical[idx['dec']]  # Historical for December
        
        interpolation_factor = calculate_interpolation_factor(esp_september, historical[idx['sep']], historical[idx['dec']])
        
        # Calculate October and November forecasts by multiplying the historical data by interpolation factor
        forecast.append(historical[idx['oct']]* interpolation_factor)  # October forecast
        forecast.append(historical[idx['nov']]* interpolation_factor)  # November forecast
        
        #fill wiht historical data starting in December to make 24 months
        months_needed=24-len(forecast)
        start_hist_idx=idx['dec']
        forecast.extend(extended_historical[start_hist_idx:start_hist_idx+months_needed])

    # For June through December (forecast_month 6 to 12)
    else:
        # First part: ESP for corresponding month through month 21
        months_from_issue_to_sep2=22-current_month_idx
        #start_idx=forecast_month-1
        forecast.extend(esp[current_month_idx:months_from_issue_to_sep2])  # Forecast for June through September of Year 2 (months 6-21)
        
        # Interpolation for October and November of Year 2
        esp_september = esp[20]  # ESP for September year 2
    
        interpolation_factor = calculate_interpolation_factor(esp_september, historical[idx['sep']], historical[idx['dec']])
        
        # Calculate October and November forecasts for Year 2 by multiplying the historical data by interpolation factor
        forecast.append(historical[idx['oct']]* interpolation_factor)  # October forecast
        forecast.append(historical[idx['nov']]* interpolation_factor)  # November forecast
        
        # Remaining months (December through May of Year 2) are historical
        months_needed=24-len(forecast)
        start_hist_idx=idx['dec']
        forecast.extend(extended_historical[start_hist_idx:start_hist_idx+months_needed])

    return forecast

# Example: Load the historical data (Powell column only for historical values)
path='/home/paige.becker/24MS/'
historical_df = pd.read_csv(path+'HistoricalMedian.csv')  # Replace with the path to your historical CSV file

# Define the directory where ESP files are stored
esp_directory = '/home/paige.becker/24MS/YDLC2/'  # Replace with the directory containing your ESP files
#options are: BMDC2, CLSC2, DRGC2, GBRW4, GLDA3, GRNU1, MPSC2, NVRN5, TPIC2, VCRC2, YDLC2

# Loop through each year from 1981 to 2020
# for year in range(1981, 2021):
#     print(f"Forecast for year {year}:")
#     # Loop through each month (1-12) for the current year
#     for month in range(1, 13):
#         forecast = generate_forecast(month, year, historical_df, esp_directory)
#         if forecast:
#             print(f"  Forecast for month {month}: {forecast}")


In [94]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

def generate_date_labels(start_year, start_month, num_months=24):
    start_date = datetime(start_year, start_month, 1)
    return [(start_date + relativedelta(months=i)).strftime("%Y-%m") for i in range(num_months)]


In [95]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta
## UPDATE ESP_DIRECTORY AND OUTPUT DIRECTORY
# Assuming `generate_forecast()` and `generate_date_labels()` are defined

# Example: Load historical Powell values
historical_df = pd.read_csv('/home/paige.becker/24MS/HistoricalMedian.csv')  # Update path
esp_directory = '/home/paige.becker/24MS/YDLC2/'  # Update path #options are: BMDC2, CLSC2, DRGC2, GBRW4, GLDA3, GRNU1, MPSC2, NVRN5, TPIC2, VCRC2, YDLC2
output_directory = '/home/paige.becker/24MS/YDLC2/MostProbable/'  # Update path

os.makedirs(output_directory, exist_ok=True)

# Loop through each year and forecast month
for year in range(1981, 2021):
    for month in range(1, 13):
        forecast = generate_forecast(month, year, historical_df, esp_directory)
        if forecast:
            # Generate date labels
            date_labels = generate_date_labels(year, month, num_months=24)

            # Create DataFrame
            df = pd.DataFrame({
                'Date': date_labels,
                'Forecast': forecast
            })

            # Save to CSV
            filename = f'forecast_{year}_{month:02d}.csv'
            output_path = os.path.join(output_directory, filename)
            df.to_csv(output_path, index=False)

            print(f"Saved forecast to {output_path}")


Saved forecast to /home/paige.becker/24MS/YDLC2/MostProbable/forecast_1981_01.csv
Saved forecast to /home/paige.becker/24MS/YDLC2/MostProbable/forecast_1981_02.csv
Saved forecast to /home/paige.becker/24MS/YDLC2/MostProbable/forecast_1981_03.csv
Saved forecast to /home/paige.becker/24MS/YDLC2/MostProbable/forecast_1981_04.csv
Saved forecast to /home/paige.becker/24MS/YDLC2/MostProbable/forecast_1981_05.csv
Saved forecast to /home/paige.becker/24MS/YDLC2/MostProbable/forecast_1981_06.csv
Saved forecast to /home/paige.becker/24MS/YDLC2/MostProbable/forecast_1981_07.csv
Saved forecast to /home/paige.becker/24MS/YDLC2/MostProbable/forecast_1981_08.csv
Saved forecast to /home/paige.becker/24MS/YDLC2/MostProbable/forecast_1981_09.csv
Saved forecast to /home/paige.becker/24MS/YDLC2/MostProbable/forecast_1981_10.csv
Saved forecast to /home/paige.becker/24MS/YDLC2/MostProbable/forecast_1981_11.csv
Saved forecast to /home/paige.becker/24MS/YDLC2/MostProbable/forecast_1981_12.csv
Saved forecast t

Now do for Probable Minimum

In [96]:
#FIRST UPDATE SITE NAMES AND DIRECTORIES
#LINES 80,83, 210

def calculate_factors(esp10_july, hist50_july, hist25_october, hist50_october):
    """
    Calculate July and October-based interpolation factors.
    """
    factor_july = esp10_july / hist50_july
    factor_october = hist25_october / hist50_october
    return factor_july, factor_october

def interpolate_factors(factor_july, factor_october):
    """
    Linearly interpolate between July and October factors to get factors for August and September.
    """
    factor_august = factor_july + (factor_october - factor_july) * (1/3)
    factor_september = factor_july + (factor_october - factor_july) * (2/3)
    return factor_august, factor_september



#second interpolation factor is year two october and november using historical 25 of september and historical 50th for december
def calculate_interpolation_factor2(hist25_sept, hist50_sept, hist50_december):
    """
    Calculate interpolation factor based on the historical 25th of July,
    and December Historical 50th values.
    """
    factor2 = (hist25_sept / hist50_sept) * (hist50_december/hist50_december)
    return factor2

#thrid interpolation factor using esp10 and 25th percentile but for october and november
def calculate_factors3(esp10_sep3, hist50_sep, hist25_december, hist50_december):
    """
    Calculate July and October-based interpolation factors.
    """
    factor_sep3 = esp10_sep3 / hist50_sep
    factor_dec = hist25_december / hist50_december
    return factor_sep3, factor_dec

def interpolate_factors3(factor_sep3, factor_dec):
    """
    Linearly interpolate between July and October factors to get factors for August and September.
    """
    factor_october3 = factor_sep3 + (factor_dec - factor_sep3) * (1/3)
    factor_november3 = factor_sep3 + (factor_dec - factor_sep3) * (2/3)
    return factor_october3, factor_november3

# Function to generate the forecast based on the rules
def generate_forecast(forecast_month, year, historical50_df,historical25_df, esp_directory):
    """
    Generate forecast based on the forecast month, ESP data, and Powell historical data.
    
    forecast_month: Month when forecast is issued (1-12).
    year: Year of the forecast (e.g., 1981, 1982, ..., 2020).
    historical_df: Historical DataFrame containing Powell historical data.
    esp_directory: Path to the directory containing ESP files.
    """
    
    # Construct the filename for the ESP file for the given year and month
    esp_filename = f'{year}-{str(forecast_month).zfill(2)}-01_percentile.csv'
    esp_filepath = os.path.join(esp_directory, esp_filename)
    
    # Load the ESP data from the file
    try:
        esp_df = pd.read_csv(esp_filepath)
    except FileNotFoundError:
        print(f"Error: ESP file '{esp_filename}' not found.")
        return []

    # Extract the 50th Percentile ESP values
    esp50 = esp_df['50th Percentile'].values
    # Extract the 10th Percentile ESP values
    esp10 = esp_df['10th Percentile'].values
    
    
    # Extract the historical 25th and 50th data
    #site options are: 'Powell', 'Fontenelle', 'Flaming Gorge', 'BlueMesa',
       #'MorrowPoint', 'Crystal ', 'TaylorPark ', 'Vallecito', 'Navajo (Mod)',
       #'Animas at Durango', 'GJ Local', 'Yampa Maybell Plus Lily'
    historical25 = historical25_df['Yampa Maybell Plus Lily'].values
    extended_historical=np.tile(historical25,2)
    forecast = []
    historical50=historical50_df['Yampa Maybell Plus Lily'].values

    #month index mapping because historical starts in October
    idx={'oct': 0, 'nov': 1, 'dec': 2,
        'jan': 3, 'feb': 4, 'mar': 5,
        'apr': 6, 'may': 7, 'jun': 8,
        'jul': 9, 'aug': 10, 'sep': 11}
    #track current month index in 0-based (jan =1, dec =12)
    current_month_idx=forecast_month-1
    # For January
    if forecast_month == 1:

        # First part: ESP50 for corresponding month through month 3
        
        months_from_issue_to_mar=0-current_month_idx
        forecast.extend(esp50[current_month_idx:3]) #through ESP march
        
        #months april through july are from ESP 10 
        forecast.extend(esp10[3:7]) #april is index 3, july is index 6 so this tells it to go up to but not include august (index 7)
	    
        # Interpolation for august and september
        esp10_july = esp10[6]  # ESP for july
                
        factor_july, factor_october = calculate_factors(
        esp10_july,
        historical50[idx['jul']],
        historical25[idx['oct']],
        historical50[idx['oct']]
        )

        factor_aug, factor_sep = interpolate_factors(factor_july, factor_october)

        # Apply interpolated factors to August and September
        forecast.append(historical50[idx['aug']] * factor_aug)  # August
        forecast.append(historical50[idx['sep']] * factor_sep)  # September

        #fill october through september of next month with historical 25th values
         #to include september
        forecast.extend(extended_historical[idx['oct']:idx['sep']+1])
        #now do second interpolation  hist25_sept, hist50_sept, hist50_december      
        hist25_sept=historical25[idx['sep']]
        hist50_sept=historical50[idx['sep']]
        hist50_dec=historical50[idx['dec']]      
        interpolation_factor2=calculate_interpolation_factor2(historical25[idx['sep']],historical50[idx['sep']],historical50[idx['dec']])
        #calculate october and november forecasts
        forecast.append(historical50[idx['oct']]*interpolation_factor2)
        forecast.append(historical50[idx['nov']]*interpolation_factor2)
        #last month is december 50th historical
        forecast.append(historical50[idx['dec']])

    #now do April
    if forecast_month==4:
        
        #first 4 months are from ESP 10
        #months april through july are from ESP 10 
        #start_idx=forecast_month-1
        forecast.extend(esp10[1:5])
        #forecast.extend(esp10[3:7]) #

        #apply interpolation for August and September
        esp10_july = esp10[4]
        factor_july, factor_october = calculate_factors(
            esp10_july,
            historical50[idx['jul']],
            historical25[idx['oct']],
            historical50[idx['oct']]
        )

        factor_aug, factor_sep = interpolate_factors(factor_july, factor_october)

        forecast.append(historical50[idx['aug']] * factor_aug)  # August
        forecast.append(historical50[idx['sep']] * factor_sep)  # September

        # --- Part 4: Append Historical 25th OctSep (Year 2 base) ---
        forecast.extend(extended_historical[idx['oct']:idx['sep']+1])  # Oct to Sep (12 values)

        # --- Part 5: Scale Oct & Nov Year 2 using dryness in September ---
        hist25_sept = historical25[idx['sep']]
        hist50_sept = historical50[idx['sep']]
        hist50_dec = historical50[idx['dec']]
        interpolation_factor2 = calculate_interpolation_factor2(hist25_sept, hist50_sept, hist50_dec)

        forecast.append(historical50[idx['oct']] * interpolation_factor2)  # October (Year 2)
        forecast.append(historical50[idx['nov']] * interpolation_factor2)  # November (Year 2)
    
        # Now add december through March unscaled
        forecast.extend([extended_historical[idx['dec']],
                        extended_historical[idx['jan']],
                        extended_historical[idx['feb']],
                        extended_historical[idx['mar']]]) #upto but not including april
    
    if forecast_month == 8 or forecast_month == 10:
        # --- Part 1: Use ESP50 from Aug through October ---
        forecast.extend(esp50[current_month_idx:current_month_idx+3])  # August september october

        # part 2: use esp10 from November through september including september
        forecast.extend(esp10[current_month_idx + 3: current_month_idx +14])

        #part 3: apply interpolation for October and November
        esp10_sep3=esp10[14]
        factor_sep3,factor_dec = calculate_factors3(
            esp10_sep3,
            historical50[idx['sep']],
            historical25[idx['dec']],
            historical50[idx['dec']])
        factor_october3, factor_november3 = interpolate_factors3(factor_sep3, factor_dec)

        forecast.append(historical50[idx['oct']]*factor_october3)
        forecast.append(historical50[idx['nov']]*factor_november3)

        #part 4: extend the forecast to be 25th percentile historical to 24 months total starting in october
        # 
        months_needed=24-len(forecast)
        start_hist_idx=idx['dec']
        forecast.extend(extended_historical[start_hist_idx:start_hist_idx+months_needed])


    return forecast


# Example: Load the historical data (Powell column only for historical values)
path='/home/paige.becker/24MS/'
historical50_df = pd.read_csv(path+'HistoricalMedian.csv') 
historical25_df = pd.read_csv(path+'Historical25.csv')
 # Replace with the path to your historical CSV file

# Define the directory where ESP files are stored
esp_directory = '/home/paige.becker/24MS/YDLC2/'  # Replace with the directory containing your ESP files
        


In [97]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

def generate_date_labels(start_year, start_month, num_months=24):
    start_date = datetime(start_year, start_month, 1)
    return [(start_date + relativedelta(months=i)).strftime("%Y-%m") for i in range(num_months)]

In [98]:

# Example: Load historical Powell values
path='/home/paige.becker/24MS/'
historical50_df = pd.read_csv(path+'HistoricalMedian.csv') 
historical25_df = pd.read_csv(path+'Historical25.csv')
 # Replace with the path to your historical CSV file

# Define the directory where ESP files are stored 
#UPDATE SITE DIRECTORIES
esp_directory = '/home/paige.becker/24MS/YDLC2/'
output_directory = '/home/paige.becker/24MS/YDLC2/MinProbable/'  # Update path

os.makedirs(output_directory, exist_ok=True)

# Loop through each year and forecast month
for year in range(1981, 2021):
    for month in range(1, 13):
        forecast = generate_forecast(month, year, historical50_df, historical25_df, esp_directory)
        if forecast:
            # Generate date labels
            date_labels = generate_date_labels(year, month, num_months=24)

            # Create DataFrame
            print(f"Length of date_labels: {len(date_labels)}, Length of forecast: {len(forecast)}")
            df = pd.DataFrame({
                'Date': date_labels,
                'Forecast': forecast
            })

            # Save to CSV
            filename = f'probable_min_{year}_{month:02d}.csv'
            output_path = os.path.join(output_directory, filename)
            df.to_csv(output_path, index=False)

            print(f"Saved forecast to {output_path}")


Length of date_labels: 24, Length of forecast: 24
Saved forecast to /home/paige.becker/24MS/YDLC2/MinProbable/probable_min_1981_01.csv
Length of date_labels: 24, Length of forecast: 24
Saved forecast to /home/paige.becker/24MS/YDLC2/MinProbable/probable_min_1981_04.csv
Length of date_labels: 24, Length of forecast: 24
Saved forecast to /home/paige.becker/24MS/YDLC2/MinProbable/probable_min_1981_08.csv
Length of date_labels: 24, Length of forecast: 24
Saved forecast to /home/paige.becker/24MS/YDLC2/MinProbable/probable_min_1981_10.csv
Length of date_labels: 24, Length of forecast: 24
Saved forecast to /home/paige.becker/24MS/YDLC2/MinProbable/probable_min_1982_01.csv
Length of date_labels: 24, Length of forecast: 24
Saved forecast to /home/paige.becker/24MS/YDLC2/MinProbable/probable_min_1982_04.csv
Length of date_labels: 24, Length of forecast: 24
Saved forecast to /home/paige.becker/24MS/YDLC2/MinProbable/probable_min_1982_08.csv
Length of date_labels: 24, Length of forecast: 24
Saved

Finally, Probable Max
Same as Probable Min, but uses 90% ESP, 75th Historical

In [132]:
#FIRST UPDATE SITE NAMES AND DIRECTORIES
#LINES 80,83, 210

def calculate_factors(esp90_july, hist50_july, hist75_october, hist50_october):
    """
    Calculate July and October-based interpolation factors.
    """
    factor_july = esp90_july / hist50_july
    factor_october = hist75_october / hist50_october
    return factor_july, factor_october

def interpolate_factors(factor_july, factor_october):
    """
    Linearly interpolate between July and October factors to get factors for August and September.
    """
    factor_august = factor_july + (factor_october - factor_july) * (1/3)
    factor_september = factor_july + (factor_october - factor_july) * (2/3)
    return factor_august, factor_september



#second interpolation factor is year two october and november using historical 25 of september and historical 50th for december
def calculate_interpolation_factor2(hist75_sept, hist50_sept, hist50_december):
    """
    Calculate interpolation factor based on the historical 25th of July,
    and December Historical 50th values.
    """
    factor2 = (hist75_sept / hist50_sept) * (hist50_december/hist50_december)
    return factor2

#thrid interpolation factor using esp10 and 25th percentile but for october and november
def calculate_factors3(esp90_sep3, hist50_sep, hist75_december, hist50_december):
    """
    Calculate July and October-based interpolation factors.
    """
    factor_sep3 = esp90_sep3 / hist50_sep
    factor_dec = hist75_december / hist50_december
    return factor_sep3, factor_dec

def interpolate_factors3(factor_sep3, factor_dec):
    """
    Linearly interpolate between July and October factors to get factors for August and September.
    """
    factor_october3 = factor_sep3 + (factor_dec - factor_sep3) * (1/3)
    factor_november3 = factor_sep3 + (factor_dec - factor_sep3) * (2/3)
    return factor_october3, factor_november3

# Function to generate the forecast based on the rules
def generate_forecast(forecast_month, year, historical50_df,historical75_df, esp_directory):
    """
    Generate forecast based on the forecast month, ESP data, and Powell historical data.
    
    forecast_month: Month when forecast is issued (1-12).
    year: Year of the forecast (e.g., 1981, 1982, ..., 2020).
    historical_df: Historical DataFrame containing Powell historical data.
    esp_directory: Path to the directory containing ESP files.
    """
    
    # Construct the filename for the ESP file for the given year and month
    esp_filename = f'{year}-{str(forecast_month).zfill(2)}-01_percentile.csv'
    esp_filepath = os.path.join(esp_directory, esp_filename)
    
    # Load the ESP data from the file
    try:
        esp_df = pd.read_csv(esp_filepath)
    except FileNotFoundError:
        print(f"Error: ESP file '{esp_filename}' not found.")
        return []

    # Extract the 50th Percentile ESP values
    esp50 = esp_df['50th Percentile'].values
    # Extract the 10th Percentile ESP values
    esp90 = esp_df['90th Percentile'].values
    
    
    # Extract the historical 25th and 50th data
    #site options are: 'Powell', 'Fontenelle', 'Flaming Gorge', 'BlueMesa',
       #'MorrowPoint', 'Crystal ', 'TaylorPark ', 'Vallecito', 'Navajo (Mod)',
       #'Animas at Durango', 'GJ Local', 'Yampa Maybell Plus Lily'
    historical75 = historical75_df['BlueMesa'].values
    extended_historical=np.tile(historical75,2)
    forecast = []
    historical50=historical50_df['BlueMesa'].values

    #month index mapping because historical starts in October
    idx={'oct': 0, 'nov': 1, 'dec': 2,
        'jan': 3, 'feb': 4, 'mar': 5,
        'apr': 6, 'may': 7, 'jun': 8,
        'jul': 9, 'aug': 10, 'sep': 11}
    #track current month index in 0-based (jan =1, dec =12)
    current_month_idx=forecast_month-1
    # For January
    if forecast_month == 1:

        # First part: ESP50 for corresponding month through month 3
        
        months_from_issue_to_mar=0-current_month_idx
        forecast.extend(esp50[current_month_idx:3]) #through ESP march
        
        #months april through july are from ESP 10 
        forecast.extend(esp90[3:7]) #april is index 3, july is index 6 so this tells it to go up to but not include august (index 7)
	    
        # Interpolation for august and september
        esp90_july = esp90[6]  # ESP for july
                
        factor_july, factor_october = calculate_factors(
        esp90_july,
        historical50[idx['jul']],
        historical75[idx['oct']],
        historical50[idx['oct']]
        )

        factor_aug, factor_sep = interpolate_factors(factor_july, factor_october)

        # Apply interpolated factors to August and September
        forecast.append(historical50[idx['aug']] * factor_aug)  # August
        forecast.append(historical50[idx['sep']] * factor_sep)  # September

        #fill october through september of next month with historical 25th values
         #to include september
        forecast.extend(extended_historical[idx['oct']:idx['sep']+1])
        #now do second interpolation  hist25_sept, hist50_sept, hist50_december      
        hist75_sept=historical75[idx['sep']]
        hist50_sept=historical50[idx['sep']]
        hist50_dec=historical50[idx['dec']]      
        interpolation_factor2=calculate_interpolation_factor2(historical75[idx['sep']],historical50[idx['sep']],historical50[idx['dec']])
        #calculate october and november forecasts
        forecast.append(historical50[idx['oct']]*interpolation_factor2)
        forecast.append(historical50[idx['nov']]*interpolation_factor2)
        #last month is december 50th historical
        forecast.append(historical50[idx['dec']])

    #now do April
    if forecast_month==4:
        
        #first 4 months are from ESP 10
        #months april through july are from ESP 10 
        #start_idx=forecast_month-1
        forecast.extend(esp90[1:5])
        #forecast.extend(esp10[3:7]) #

        #apply interpolation for August and September
        esp90_july = esp90[4]
        factor_july, factor_october = calculate_factors(
            esp90_july,
            historical50[idx['jul']],
            historical75[idx['oct']],
            historical50[idx['oct']]
        )

        factor_aug, factor_sep = interpolate_factors(factor_july, factor_october)

        forecast.append(historical50[idx['aug']] * factor_aug)  # August
        forecast.append(historical50[idx['sep']] * factor_sep)  # September

        # --- Part 4: Append Historical 25th OctSep (Year 2 base) ---
        forecast.extend(extended_historical[idx['oct']:idx['sep']+1])  # Oct to Sep (12 values)

        # --- Part 5: Scale Oct & Nov Year 2 using dryness in September ---
        hist75_sept = historical75[idx['sep']]
        hist50_sept = historical50[idx['sep']]
        hist50_dec = historical50[idx['dec']]
        interpolation_factor2 = calculate_interpolation_factor2(hist75_sept, hist50_sept, hist50_dec)

        forecast.append(historical50[idx['oct']] * interpolation_factor2)  # October (Year 2)
        forecast.append(historical50[idx['nov']] * interpolation_factor2)  # November (Year 2)
    
        # Now add december through March unscaled
        forecast.extend([extended_historical[idx['dec']],
                        extended_historical[idx['jan']],
                        extended_historical[idx['feb']],
                        extended_historical[idx['mar']]]) #upto but not including april
    
    if forecast_month == 8 or forecast_month == 10:
        # --- Part 1: Use ESP50 from Aug through October ---
        forecast.extend(esp50[current_month_idx:current_month_idx+3])  # August september october

        # part 2: use esp10 from November through september including september
        forecast.extend(esp90[current_month_idx + 3: current_month_idx +14])

        #part 3: apply interpolation for October and November
        esp90_sep3=esp90[14]
        factor_sep3,factor_dec = calculate_factors3(
            esp90_sep3,
            historical50[idx['sep']],
            historical75[idx['dec']],
            historical50[idx['dec']])
        factor_october3, factor_november3 = interpolate_factors3(factor_sep3, factor_dec)

        forecast.append(historical50[idx['oct']]*factor_october3)
        forecast.append(historical50[idx['nov']]*factor_november3)

        #part 4: extend the forecast to be 25th percentile historical to 24 months total starting in october
        # 
        months_needed=24-len(forecast)
        start_hist_idx=idx['dec']
        forecast.extend(extended_historical[start_hist_idx:start_hist_idx+months_needed])


    return forecast


# Example: Load the historical data (Powell column only for historical values)
path='/home/paige.becker/24MS/'
historical50_df = pd.read_csv(path+'HistoricalMedian.csv') 
historical75_df = pd.read_csv(path+'Historical75.csv')
 # Replace with the path to your historical CSV file

# Define the directory where ESP files are stored
esp_directory = '/home/paige.becker/24MS/BMDC2/'  # Replace with the directory containing your ESP files
        


In [133]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

def generate_date_labels(start_year, start_month, num_months=24):
    start_date = datetime(start_year, start_month, 1)
    return [(start_date + relativedelta(months=i)).strftime("%Y-%m") for i in range(num_months)]

In [134]:
# Example: Load historical Powell values
path='/home/paige.becker/24MS/'
historical50_df = pd.read_csv(path+'HistoricalMedian.csv') 
historical75_df = pd.read_csv(path+'Historical75.csv')
 # Replace with the path to your historical CSV file

# Define the directory where ESP files are stored 
#UPDATE SITE DIRECTORIES
esp_directory = '/home/paige.becker/24MS/BMDC2/'
output_directory = '/home/paige.becker/24MS/BMDC2/MaxProbable/'  # Update path

os.makedirs(output_directory, exist_ok=True)

# Loop through each year and forecast month
for year in range(1981, 2021):
    for month in range(1, 13):
        forecast = generate_forecast(month, year, historical50_df, historical75_df, esp_directory)
        if forecast:
            # Generate date labels
            date_labels = generate_date_labels(year, month, num_months=24)

            # Create DataFrame
            print(f"Length of date_labels: {len(date_labels)}, Length of forecast: {len(forecast)}")
            df = pd.DataFrame({
                'Date': date_labels,
                'Forecast': forecast
            })

            # Save to CSV
            filename = f'probable_max_{year}_{month:02d}.csv'
            output_path = os.path.join(output_directory, filename)
            df.to_csv(output_path, index=False)

            print(f"Saved forecast to {output_path}")


Length of date_labels: 24, Length of forecast: 24
Saved forecast to /home/paige.becker/24MS/BMDC2/MaxProbable/probable_max_1981_01.csv
Length of date_labels: 24, Length of forecast: 24
Saved forecast to /home/paige.becker/24MS/BMDC2/MaxProbable/probable_max_1981_04.csv
Length of date_labels: 24, Length of forecast: 24
Saved forecast to /home/paige.becker/24MS/BMDC2/MaxProbable/probable_max_1981_08.csv
Length of date_labels: 24, Length of forecast: 24
Saved forecast to /home/paige.becker/24MS/BMDC2/MaxProbable/probable_max_1981_10.csv
Length of date_labels: 24, Length of forecast: 24
Saved forecast to /home/paige.becker/24MS/BMDC2/MaxProbable/probable_max_1982_01.csv
Length of date_labels: 24, Length of forecast: 24
Saved forecast to /home/paige.becker/24MS/BMDC2/MaxProbable/probable_max_1982_04.csv
Length of date_labels: 24, Length of forecast: 24
Saved forecast to /home/paige.becker/24MS/BMDC2/MaxProbable/probable_max_1982_08.csv
Length of date_labels: 24, Length of forecast: 24
Saved